# Smart Resume Classifier + Skill Extractor

This notebook builds a beginner-friendly NLP + ML project that:
- classifies resumes into job roles
- extracts important skills
- compares a resume with a job description
- identifies missing skills


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Load dataset

In [ ]:
df = pd.read_csv('../data/resume_dataset.csv')
df.head()

## 3. Check dataset shape and class distribution

In [ ]:
print(df.shape)
df['job_role'].value_counts()

## 4. Text cleaning

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9+#./ ]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['resume_text'].apply(clean_text)
df[['resume_text', 'cleaned_text', 'job_role']].head()

## 5. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'],
    df['job_role'],
    test_size=0.2,
    random_state=42,
    stratify=df['job_role']
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

## 6. Build TF-IDF + Logistic Regression model

In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

## 7. Evaluate model

In [ ]:
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print('Accuracy:', round(acc, 4))
print()
print(classification_report(y_test, preds))

## 8. Skill extraction

In [ ]:
skills = Path('../data/skills_list.txt').read_text(encoding='utf-8').splitlines()

def extract_skills(text, skills_list):
    text = clean_text(text)
    found = []
    for skill in skills_list:
        pattern = rf'(?<!\w){re.escape(skill.lower())}(?!\w)'
        if re.search(pattern, text):
            found.append(skill)
    return sorted(set(found))

sample_resume = df['resume_text'].iloc[0]
extract_skills(sample_resume, skills)[:20]

## 9. Resume prediction example

In [ ]:
sample_text = clean_text(df['resume_text'].iloc[5])
pred_role = model.predict([sample_text])[0]
pred_role

## 10. Job description matching

In [ ]:
def jaccard_similarity(a, b):
    a, b = set(map(str.lower, a)), set(map(str.lower, b))
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

resume_text = """Python pandas numpy scikit-learn SQL machine learning dashboard data analysis streamlit git"""
job_description = """Looking for a Data Scientist with Python, SQL, machine learning, data analysis, git, and streamlit skills."""

resume_skills = extract_skills(resume_text, skills)
jd_skills = extract_skills(job_description, skills)
score = jaccard_similarity(resume_skills, jd_skills)

print('Resume skills:', resume_skills)
print('JD skills:', jd_skills)
print('Match score:', round(score, 2))

## 11. Skill gap analysis

In [ ]:
matched = sorted(set(map(str.lower, resume_skills)) & set(map(str.lower, jd_skills)))
missing = sorted(set(map(str.lower, jd_skills)) - set(map(str.lower, resume_skills)))
extra = sorted(set(map(str.lower, resume_skills)) - set(map(str.lower, jd_skills)))

print('Matched skills:', matched)
print('Missing skills:', missing)
print('Extra skills:', extra)

## 12. Conclusion

This project demonstrates a complete beginner NLP + ML workflow:
- text preprocessing
- TF-IDF feature extraction
- Logistic Regression classification
- skill extraction
- similarity scoring
- simple skill gap analysis

It can be extended with a larger Kaggle dataset, more job roles, better skill extraction, and deployment using Streamlit.
